# `HR_TRG_LOC_INSERT.txt` Conversion to Spark SQL

**Conversion Timestamp:** 2024-07-30 12:00:00 UTC

This notebook converts the ODI SQL for loading `TRG_LOC` table from `LOCATIONS` table into Databricks Spark SQL. It includes parameter setup, data insertion, and target optimization.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., F for Full, I for Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "5. ETL Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "2099-12-31 23:59:59", "6. ETL Current Extract Time (YYYY-MM-DD HH:MM:SS)")

# ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {10}: Setup ETL parameter views

CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id,
  CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

# Insert into Target Table `trg_loc`

In [ ]:
%sql
-- SCEN_TASK_NO {20}: Optional: Truncate target table for full reload
-- This step assumes a full reload scenario, typical with direct INSERTs in ODI.
TRUNCATE TABLE workspace.hr.trg_loc;

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert data into HR.TRG_LOC

INSERT INTO workspace.hr.trg_loc
(
  location_id ,
  street_address ,
  postal_code ,
  city ,
  state_province ,
  country_id
)
SELECT
  locations.location_id ,
  locations.street_address ,
  locations.postal_code ,
  locations.city ,
  locations.state_province ,
  locations.country_id
FROM
  workspace.hr.locations AS locations;

In [ ]:
%sql
SELECT COUNT(*) AS record_count FROM workspace.hr.trg_loc;

# Optimize Target Table

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;

-- Assuming LOCATION_ID is a primary key or frequently queried column for ZORDER
OPTIMIZE workspace.hr.trg_loc ZORDER BY (location_id);

# Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were created in this specific ODI task,
-- so no cleanup is explicitly required here.
SELECT 'No temporary tables to drop' AS cleanup_status;

# Validation

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** All Oracle schema references (`HR`) have been converted to `workspace.hr` and table names to lowercase (`trg_loc`, `locations`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` Oracle hint has been removed as it's not applicable in Databricks Delta Lake.
3.  **Target Truncation:** An optional `TRUNCATE TABLE` statement has been added before the `INSERT` statement. This is a common pattern in ODI for full loads when the `INSERT` directly populates a target table, interpreting `SCEN_TASK_NO {20}` as a potential cleanup step. Adjust this if the load is meant to be an incremental append.
4.  **Data Types:** The `INSERT` statement does not specify data types directly. It is assumed that the `workspace.hr.trg_loc` table is created with compatible Spark SQL data types (e.g., `STRING` for `VARCHAR2`, `BIGINT` for `NUMBER(p,0)` if applicable, `TIMESTAMP` for `DATE`/`TIMESTAMP`) based on the source `workspace.hr.locations` table. Ensure `TRG_LOC` DDL is consistent.
5.  **Parameter Widgets:** Placeholder widgets for `ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`, `ETL_LAST_EXTRACT_TIME`, and `ETL_CURRENT_EXTRACT_TIME` have been included. The `INSERT` statement itself does not use these, but they are standard for ODI conversions.
6.  **Optimization:** An `OPTIMIZE ... ZORDER BY` statement has been added for `workspace.hr.trg_loc`. `location_id` was chosen as a ZORDER key, but this should be reviewed and adjusted based on actual query patterns for the `trg_loc` table.
7.  **Error Handling/Staging/Flow Tables:** The original SQL only contained a direct `INSERT` statement. Therefore, explicit staging (`C$`), flow (`I$`), and error (`E$`) table creation/population, and associated PK violation checks, were not generated. If the original ODI process had these steps, they would need to be added manually based on the full ODI session details.